In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import os
from pathlib import Path
from openpyxl import load_workbook

In [3]:
root = "./var/WAT"

In [4]:
os.listdir(root)

['.DS_Store',
 'DAHO (Kharkiv)',
 'DACHKO (Cherkassy)',
 'DAHEO (Kherson)',
 'DAHMO (Khmelnitsky)',
 'DAOO (Odessa)']

In [5]:
def list_files(directory: str, suffix: str) -> list[Path]:
    """Recursively list only files in `directory` with the given `suffix`."""
    return [p for p in Path(directory).rglob(f"*{suffix}") if p.is_file()]

In [6]:
archives = list_files(root, "xlsx")
archives

[PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-archive-20250719.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-R-wiki-20250908.xlsx'),
 PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-D-wiki-20250817a.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-R-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DACHKO (Cherkassy)/DACHKO-D-wiki-20250904.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-archive-20250819.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-P-wiki-20250708.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-wiki-20250821.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-R-wiki-20250724.xlsx'),
 PosixPath('var/WAT/DAHEO (Kherson)/DAHEO-D-archive-20250707.xlsx'),
 PosixPath('var/WAT/DAHMO (Khmelnitsky)/DAHMO-D-wiki-20250910.xlsx'),
 PosixPath('var/WAT/DAHMO (Khmelnitsky)/DAHMO-K-wiki-20250820.xlsx'),
 PosixPath('var/WAT/DAHMO (Khmelnitsky)/DAHMO-R-wiki-20250908.xlsx'),
 PosixPath('var/WAT/DAHMO (Khmelnitsky)/DAHMO-P-wiki-20250731.xlsx'),
 PosixPath('var/WAT/DAOO (Odessa)/DAOO-D-wiki-2025

In [7]:
archives[0]

PosixPath('var/WAT/DAHO (Kharkiv)/DAHO-archive-20250719.xlsx')

In [8]:
wb = load_workbook(archives[0])

In [66]:
ws = wb.worksheets[1]

In [67]:
ws.title

'fund 958'

In [68]:
def find_table_header_row(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return ws[c.row + 1]
    return None

In [69]:
def find_formula_row_index(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return c.row
    return None

In [70]:
def max_content_col(row):
    for cell in reversed(row):
        if cell.value:
            return cell.column_letter
    return row[0].column_letter

In [71]:
def partition_sheet(ws):
    result = {}
    f_row = find_formula_row_index(ws)
    if f_row is not None:
        h_row = f_row + 1
        result["table_header"] = f"A{h_row}:{max_content_col(ws[h_row])}{h_row}"
        max_col = "A"
        for row in range(1, f_row-1):
            max_col = max(max_col, max_content_col(ws[row]))
        result["sheet_header"] = f"A1:{max_col}{f_row-1}"
        max_col = "A"
        for row in range((h_row+1), ws.max_row):
            max_col = max(max_col, max_content_col(ws[row]))
            if not ws[row][0].value:
                result["table_content"] = f"A{h_row+1}:{max_col}{row-1}"
                break
    return result

In [72]:
p=partition_sheet(ws)

In [73]:
def extract_strings_in_range(ws, cell_range):
    result = set()
    for row in ws[cell_range]:
        for cell in row:
            if cell.value:
                result.add(cell.value)
    return result

In [74]:
def extract_strings_by_partition(ws):
    return { key: extract_strings_in_range(ws, cell_range) 
             for key, cell_range in partition_sheet(ws).items() }

In [75]:
strings = extract_strings_by_partition(ws)

In [76]:
strings["sheet_header"]

{'26 May 2025',
 '3 Aug 2023',
 '9 May 2022',
 '958',
 'DAHO',
 'Fund link(s):',
 'Office of the Kharkiv Provincial Rabbi, Kharkiv',
 'accessed:',
 'archive',
 'change date:',
 'http://archives.kh.gov.ua/?page_id=24086',
 'page said updated:',
 'source (page gone now):'}

In [77]:
strings["table_header"]

{'Acquired files transcribed',
 'Availability',
 'Comments',
 'Date(s)',
 'Future examination',
 'Opus #, link',
 'Opus description, link',
 'Other acquisitions backlog',
 'Other files processed',
 'Other files to acquire',
 'Other pages processed',
 'Priority acquisition backlog',
 'Priority files processed',
 'Priority files to acquire',
 'Priority pages processed'}

In [65]:
len(strings["table_content"])

1481

In [ ]:
all_headers = {}
for i, archive in enumerate(archives):
    print(i, archive)
    wb = load_workbook(archive)
    for sheet in wb.worksheets:
        row = find_table_header_row(sheet)
        if row:
            for cell in row:
                value = cell.value
                if value:
                    entry = all_headers.get(value, [])
                    entry.append(sheet.title)
                    all_headers[value] = entry

In [ ]:
for header, entry in all_headers.items():
    print(header, len(entry))

In [ ]:
ws["A1:B5"]

In [ ]:
c = ws["A1"]

In [55]:
p

{'table_header': 'A8:O8', 'sheet_header': 'A1:O6', 'table_content': 'A9:O583'}

In [54]:
for row in ws[p["table_header"]]:
    for cell in row: 
        print(cell.value)

Fund #, link
Fund description, link
Date(s)
Availability
Priority files to aquire
Priority files processed
Priority pages processed
Priority acquisitions backlog
Other files to aquire
Other files processed
Other pages processed
Other acquisitions backlog
Acquired files transcribed
Future examination
Comments


In [ ]:
range(10,0,-1)

In [ ]:
for i in range(5,0,-1):
    print(i)

In [43]:
ws.max_row

603

In [79]:
x=set([1,2,3])

In [80]:
x

{1, 2, 3}

In [81]:
x.add(set(range(6,9)))

TypeError: unhashable type: 'set'

In [82]:
x.union(set(range(6,9)))

{1, 2, 3, 6, 7, 8}

In [83]:
x


{1, 2, 3}